# DVF 2022 — Exploration d'une mutation à Abbaretz

Ce notebook explore une mutation du fichier DVF 2022 afin d'observer les lignes qu'elle produit et la valeur foncière qui leur est associée. Il porte sur la commune d'Abbaretz (Loire-Atlantique, 44), à la date du 22 juin 2022.

## Cellule 1 — Connexion au fichier

Le fichier est interrogé avec DuckDB, directement au format Parquet, sans le charger entièrement en mémoire. Le chemin pointe vers le fichier `dvf-2022.parquet` placé dans le dossier `data/`.

In [3]:
import duckdb
from pathlib import Path

# Chemin vers le fichier Parquet, place dans le dossier data/
FICHIER = Path(r"./data/dvf-2022.parquet")

# Connexion DuckDB
con = duckdb.connect()
pq = str(FICHIER)

# Verification
assert FICHIER.exists(), f"Fichier introuvable : {FICHIER}"
nb_lignes = con.execute(f"SELECT count(*) FROM '{pq}'").fetchone()[0]
print(f"Fichier : {FICHIER.name}")
print(f"Lignes  : {nb_lignes:,}".replace(',', ' '))

Fichier : dvf-2022.parquet
Lignes  : 4 617 590


## Cellule 2 — Lignes enregistrées à Abbaretz le 22 juin 2022

Sélection de toutes les lignes de cette commune à cette date. Le tableau affiche, pour chaque ligne, le prix, le type de local, la surface et l'adresse.

In [4]:
lignes = con.execute(f"""
    SELECT "Date mutation", "Valeur fonciere", "Type local",
           "Surface reelle bati", "Nombre pieces principales",
           "No voie", "Voie", "Code postal", "Surface terrain"
    FROM '{pq}'
    WHERE "Commune" = 'ABBARETZ'
      AND "Date mutation" = '2022-06-22'
    ORDER BY "Surface reelle bati"
""").fetchdf()

print(f"Nombre de lignes : {len(lignes)}")
lignes

Nombre de lignes : 6


,Date mutation,Valeur fonciere,Type local,Surface reelle bati,Nombre pieces principales,No voie,Voie,Code postal,Surface terrain
0,2022-06-22,94500,Maison,70,3,25,DES COQUELICOTS,44170,458
1,2022-06-22,94500,Maison,70,3,25,DES COQUELICOTS,44170,458
2,2022-06-22,94500,Maison,70,3,25,DES COQUELICOTS,44170,458
3,2022-06-22,94500,Maison,70,3,25,DES COQUELICOTS,44170,458
4,2022-06-22,94500,Maison,87,4,25,DES COQUELICOTS,44170,458
5,2022-06-22,94500,Maison,87,4,25,DES COQUELICOTS,44170,458


## Cellule 3 — Valeurs observées

Trois relevés factuels sur ces lignes : le nombre de valeurs de prix distinctes, les surfaces bâties présentes, et la répartition des lignes par type et surface.

In [5]:
prix_distincts = sorted(lignes["Valeur fonciere"].unique())
surfaces = sorted(x for x in lignes["Surface reelle bati"].unique())

print(f"Valeur(s) fonciere(s) distincte(s) : {prix_distincts}")
print(f"Surface(s) batie(s) distincte(s)   : {surfaces}")
print()

repartition = (
    lignes.groupby(["Type local", "Surface reelle bati", "Nombre pieces principales"], dropna=False)
          .size()
          .reset_index(name="Nb de lignes")
)
repartition

Valeur(s) fonciere(s) distincte(s) : [np.int32(94500)]
Surface(s) batie(s) distincte(s)   : [np.int64(70), np.int64(87)]



,Type local,Surface reelle bati,Nombre pieces principales,Nb de lignes
0,Maison,70,3,4
1,Maison,87,4,2
